# 2.1 — Comparaison et choix du modèle

Chaque modèle est entraîné **1 epoch** sur le split train (6 479 images) puis évalué sur le split val (1 458 images).

In [ ]:
%pip install ultralytics torchvision transformers torchmetrics -q

In [ ]:
import torch, time, numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchmetrics.detection.mean_ap import MeanAveragePrecision

DATASET_ROOT = Path(r"C:\Users\luigi\OneDrive\Documents\Ecole 18.06\Traitement_image\Projet\SH17dataset")
YAML_PATH    = DATASET_ROOT / 'sh17.yaml'
NC           = 17
EPOCHS       = 1   # objectif initial : 10 epochs — réduit à 1 en raison de coupures WiFi et kernels surchargés (voir doc 02)
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RESULTS      = {}

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

In [ ]:
# ---- Dataset partagé (Faster R-CNN et SSD) ----
class SH17Detection(Dataset):
    def __init__(self, img_dir, lbl_dir, size=640):
        all_imgs = sorted(Path(img_dir).glob('*.*'))
        self.lbl_dir = Path(lbl_dir)
        self.size = size
        # Garde seulement les images avec annotations non vides
        self.imgs = [
            p for p in all_imgs
            if (self.lbl_dir / (p.stem + '.txt')).exists()
            and (self.lbl_dir / (p.stem + '.txt')).stat().st_size > 0
        ]
        self.tf = T.Compose([T.Resize((size, size)), T.ToTensor()])

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        img = Image.open(img_path).convert('RGB')
        # Les coordonnées YOLO sont normalisées : on les met à l'échelle de
        # l'image APRÈS resize (self.size x self.size), pas de l'image d'origine,
        # sinon les boîtes ne correspondent plus au tenseur vu par le modèle.
        w, h = self.size, self.size
        boxes, labels = [], []
        for line in (self.lbl_dir / (img_path.stem + '.txt')).read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            x1 = max(0., (cx - bw / 2) * w)
            y1 = max(0., (cy - bh / 2) * h)
            x2 = min(w,  (cx + bw / 2) * w)
            y2 = min(h,  (cy + bh / 2) * h)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(c + 1)  # 0 = fond
        target = {
            'boxes':  torch.tensor(boxes,  dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
        }
        return self.tf(img), target


def collate_fn(batch):
    return tuple(zip(*batch))


# ---- Boucle d'entraînement (Faster R-CNN et SSD) ----
def train_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0
    for imgs, targets in loader:
        imgs    = [i.to(device) for i in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        losses  = sum(model(imgs, targets).values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total += losses.item()
    return total / len(loader)


# ---- Évaluation mAP50 (Faster R-CNN et SSD) ----
def eval_map50(model, loader, device):
    model.eval()
    metric = MeanAveragePrecision(iou_thresholds=[0.5])
    with torch.no_grad():
        for imgs, targets in loader:
            imgs  = [i.to(device) for i in imgs]
            preds = [{k: v.cpu() for k, v in p.items()} for p in model(imgs)]
            tgts  = [{'boxes': t['boxes'], 'labels': t['labels']} for t in targets]
            metric.update(preds, tgts)
    return float(metric.compute()['map_50'])


# ---- Temps d'inférence (Faster R-CNN et SSD) ----
def measure_inference(model, dataset, device, n=50):
    model.eval()
    times = []
    with torch.no_grad():
        for i in range(min(n, len(dataset))):
            img, _ = dataset[i]
            if device == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model([img.to(device)])
            if device == 'cuda':
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times))


print('Utilitaires chargés.')

## Modèle 1 — YOLOv8m

In [ ]:
from ultralytics import YOLO

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()

yolo = YOLO('yolov8m.pt')
yolo.train(
    data=str(YAML_PATH), epochs=EPOCHS, imgsz=640, batch=16,
    device=0 if DEVICE == 'cuda' else 'cpu',
    verbose=False, plots=False, save=False
)
val_res = yolo.val(
    data=str(YAML_PATH), imgsz=640,
    device=0 if DEVICE == 'cuda' else 'cpu',
    verbose=False
)

train_min = (time.time() - t0) / 60
gpu_gb    = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0

# Temps d'inférence
val_paths = list((DATASET_ROOT / 'images' / 'val').glob('*.*'))[:50]
inf_times = []
for p in val_paths:
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    yolo.predict(str(p), verbose=False)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    inf_times.append((time.perf_counter() - t1) * 1000)

RESULTS['YOLOv8m'] = {
    'mAP50':      round(val_res.box.map50 * 100, 1),
    'inf_ms':     round(np.mean(inf_times), 1),
    'gpu_gb':     round(gpu_gb, 1),
    'train_min':  round(train_min, 1),
}
print('YOLOv8m :', RESULTS['YOLOv8m'])

## Modèle 2 — Faster R-CNN (ResNet50-FPN)

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FastRCNNPredictor

train_ds_fr = SH17Detection(DATASET_ROOT/'images'/'train', DATASET_ROOT/'labels'/'train', size=640)
val_ds_fr   = SH17Detection(DATASET_ROOT/'images'/'val',   DATASET_ROOT/'labels'/'val',   size=640)
train_dl_fr = DataLoader(train_ds_fr, batch_size=4, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_dl_fr   = DataLoader(val_ds_fr,   batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=0)

model_fr = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
in_feat  = model_fr.roi_heads.box_predictor.cls_score.in_features
model_fr.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NC + 1)
model_fr.to(DEVICE)
opt_fr = torch.optim.AdamW(model_fr.parameters(), lr=1e-4, weight_decay=1e-4)

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()

for ep in range(EPOCHS):
    loss = train_epoch(model_fr, train_dl_fr, opt_fr, DEVICE)
    print(f'  Faster R-CNN  epoch {ep+1:02d}/{EPOCHS}  loss={loss:.3f}')

map50     = eval_map50(model_fr, val_dl_fr, DEVICE)
train_min = (time.time() - t0) / 60
gpu_gb    = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
inf_ms    = measure_inference(model_fr, val_ds_fr, DEVICE)

RESULTS['Faster R-CNN'] = {
    'mAP50':     round(map50 * 100, 1),
    'inf_ms':    round(inf_ms, 1),
    'gpu_gb':    round(gpu_gb, 1),
    'train_min': round(train_min, 1),
}
print('Faster R-CNN :', RESULTS['Faster R-CNN'])

## Modèle 3 — SSD300 (VGG16)

In [ ]:
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection._utils import retrieve_out_channels

train_ds_ssd = SH17Detection(DATASET_ROOT/'images'/'train', DATASET_ROOT/'labels'/'train', size=300)
val_ds_ssd   = SH17Detection(DATASET_ROOT/'images'/'val',   DATASET_ROOT/'labels'/'val',   size=300)
train_dl_ssd = DataLoader(train_ds_ssd, batch_size=8, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_dl_ssd   = DataLoader(val_ds_ssd,   batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=0)

model_ssd    = ssd300_vgg16(weights='DEFAULT')
in_channels  = retrieve_out_channels(model_ssd.backbone, (300, 300))
num_anchors  = model_ssd.anchor_generator.num_anchors_per_location()
model_ssd.head.classification_head = SSDClassificationHead(
    in_channels=in_channels, num_anchors=num_anchors, num_classes=NC + 1
)
model_ssd.to(DEVICE)
opt_ssd = torch.optim.AdamW(model_ssd.parameters(), lr=1e-4, weight_decay=1e-4)

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()

for ep in range(EPOCHS):
    loss = train_epoch(model_ssd, train_dl_ssd, opt_ssd, DEVICE)
    print(f'  SSD  epoch {ep+1:02d}/{EPOCHS}  loss={loss:.3f}')

map50_ssd = eval_map50(model_ssd, val_dl_ssd, DEVICE)
train_min = (time.time() - t0) / 60
gpu_gb    = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
inf_ms    = measure_inference(model_ssd, val_ds_ssd, DEVICE)

RESULTS['SSD300'] = {
    'mAP50':     round(map50_ssd * 100, 1),
    'inf_ms':    round(inf_ms, 1),
    'gpu_gb':    round(gpu_gb, 1),
    'train_min': round(train_min, 1),
}
print('SSD300 :', RESULTS['SSD300'])

## Modèle 4 — DETR (ResNet50)

In [ ]:
import gc
# Libère la VRAM des modèles précédents avant de charger DETR
for var in ['yolo', 'model_fr', 'model_ssd']:
    if var in dir():
        del var
gc.collect()
torch.cuda.empty_cache()

from transformers import DetrForObjectDetection, DetrImageProcessor

processor_detr = DetrImageProcessor.from_pretrained('facebook/detr-resnet-50')


class SH17Detr(Dataset):
    """Dataset pour DETR — annotations converties au format COCO pixel."""
    def __init__(self, img_dir, lbl_dir):
        all_imgs = sorted(Path(img_dir).glob('*.*'))
        self.lbl_dir = Path(lbl_dir)
        self.imgs = [
            p for p in all_imgs
            if (self.lbl_dir / (p.stem + '.txt')).exists()
            and (self.lbl_dir / (p.stem + '.txt')).stat().st_size > 0
        ]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_path = self.imgs[idx]
        img = Image.open(img_path).convert('RGB')
        iw, ih = img.size
        annotations = []
        for line in (self.lbl_dir / (img_path.stem + '.txt')).read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            c = int(parts[0])
            cx, cy, bw, bh = map(float, parts[1:5])
            annotations.append({
                'bbox':        [(cx - bw/2)*iw, (cy - bh/2)*ih, bw*iw, bh*ih],
                'category_id': c,
                'area':        bw*iw * bh*ih,
                'iscrowd':     0,
            })
        return img, {'image_id': idx, 'annotations': annotations}


def collate_detr(batch):
    imgs, targets = zip(*batch)
    return processor_detr(images=list(imgs), annotations=list(targets), return_tensors='pt')


train_ds_detr = SH17Detr(DATASET_ROOT/'images'/'train', DATASET_ROOT/'labels'/'train')
val_ds_detr   = SH17Detr(DATASET_ROOT/'images'/'val',   DATASET_ROOT/'labels'/'val')
train_dl_detr = DataLoader(train_ds_detr, batch_size=1, shuffle=True,  collate_fn=collate_detr, num_workers=0)
val_dl_detr   = DataLoader(val_ds_detr,   batch_size=2, shuffle=False, collate_fn=collate_detr, num_workers=0)
print(f'DETR datasets : train={len(train_ds_detr)}  val={len(val_ds_detr)}')

In [ ]:
model_detr = DetrForObjectDetection.from_pretrained(
    'facebook/detr-resnet-50', num_labels=NC, ignore_mismatched_sizes=True
).to(DEVICE)
opt_detr = torch.optim.AdamW(model_detr.parameters(), lr=1e-4, weight_decay=1e-4)
scaler   = torch.cuda.amp.GradScaler()

if DEVICE == 'cuda':
    torch.cuda.reset_peak_memory_stats()
t0 = time.time()

for ep in range(EPOCHS):
    model_detr.train()
    total = 0.0
    for batch in train_dl_detr:
        pv  = batch['pixel_values'].to(DEVICE)
        pm  = batch['pixel_mask'].to(DEVICE)
        lbs = [{k: v.to(DEVICE) for k, v in t.items()} for t in batch['labels']]
        opt_detr.zero_grad()
        with torch.cuda.amp.autocast():
            out = model_detr(pixel_values=pv, pixel_mask=pm, labels=lbs)
        scaler.scale(out.loss).backward()
        scaler.step(opt_detr)
        scaler.update()
        total += out.loss.item()
    print(f'  DETR  epoch {ep+1:02d}/{EPOCHS}  loss={total/len(train_dl_detr):.3f}')

# Évaluation mAP50
model_detr.eval()
metric_detr = MeanAveragePrecision(iou_thresholds=[0.5])
with torch.no_grad():
    for batch in val_dl_detr:
        pv   = batch['pixel_values'].to(DEVICE)
        pm   = batch['pixel_mask'].to(DEVICE)
        with torch.cuda.amp.autocast():
            out  = model_detr(pixel_values=pv, pixel_mask=pm)
        sizes = torch.tensor([[pv.shape[-2], pv.shape[-1]]] * pv.shape[0])
        dets  = processor_detr.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        preds = [{'boxes': d['boxes'].cpu(), 'scores': d['scores'].cpu(), 'labels': d['labels'].cpu()} for d in dets]
        tgts  = [{'boxes': t['boxes'].cpu(), 'labels': t['class_labels'].cpu()} for t in batch['labels']]
        metric_detr.update(preds, tgts)
map50_detr = float(metric_detr.compute()['map_50'])

train_min = (time.time() - t0) / 60
gpu_gb    = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0

# Temps d'inférence DETR
detr_times = []
for i in range(min(50, len(val_ds_detr))):
    img, _ = val_ds_detr[i]
    enc = processor_detr(images=[img], return_tensors='pt')
    pv  = enc['pixel_values'].to(DEVICE)
    pm  = enc['pixel_mask'].to(DEVICE)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    with torch.no_grad():
        model_detr(pixel_values=pv, pixel_mask=pm)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    detr_times.append((time.perf_counter() - t1) * 1000)

RESULTS['DETR'] = {
    'mAP50':     round(map50_detr * 100, 1),
    'inf_ms':    round(np.mean(detr_times), 1),
    'gpu_gb':    round(gpu_gb, 1),
    'train_min': round(train_min, 1),
}
print('DETR :', RESULTS['DETR'])

In [ ]:
print('\n=== Comparaison des 4 modèles — 10 epochs, split val (1 458 images) ===\n')
print(f"{'Modèle':<16} {'mAP50 (%)':>10} {'Inférence (ms)':>15} {'GPU (GB)':>10} {'10 epochs (min)':>16}")
print('-' * 55)
for name, r in RESULTS.items():
    print(f"{name:<16} {r['mAP50']:>10} {r['inf_ms']:>15} {r['gpu_gb']:>10} {r['train_min']:>16}")

## Modèle retenu

**YOLOv8m** — meilleur compromis vitesse d'entraînement / performance à convergence (voir doc 02).